In [5]:
import re
import pandas as pd

def clean_verse_text(text):
    """Remove leading '. ' or '." ' or similar unwanted punctuation at the start"""
    if not text:
        return ''
    
    # Remove leading . followed by space (including cases like '". ' or '. ')
    text = re.sub(r'^\s*[\."]\s*', '', text).strip()
    # Additional cleanup: remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_verses_to_df(verses_file):
    """Extract verses: each line = one entry, carry over number if missing"""
    records = []
    last_number = None

    with open(verses_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n\r')
            stripped = line.strip()
            if not stripped:
                continue

            # Check if line starts with number.
            match = re.match(r'^\s*(\d+)\.\s*(.*)', line)
            if match:
                num = int(match.group(1))
                verses_text = match.group(2).strip()
                last_number = num
            else:
                # No number → increment from last known number
                if last_number is None:
                    last_number = 1
                else:
                    last_number += 1
                verses_text = re.sub(r'^\s*\d+\.?\s*', '', stripped).strip()

            # Clean leading '. ' or '." '
            verses_text = clean_verse_text(verses_text)

            records.append({'number': last_number, 'verses': verses_text})

    df = pd.DataFrame(records)
    return df


def extract_notes(file_path):
    """Merge multiple lines in notes with '\n'"""
    notes_dict = {}
    last_number = None
    current_lines = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n\r')
            stripped = line.strip()
            if not stripped:
                continue

            match = re.match(r'^\s*(\d+)\.\s*(.*)', line)
            if match:
                if last_number is not None and current_lines:
                    notes_dict[last_number] = '\\n'.join(current_lines)

                last_number = int(match.group(1))
                current_lines = [match.group(2).strip()]
            else:
                clean_line = re.sub(r'^\s*\d+\.?\s*', '', stripped).strip()
                if clean_line and last_number is not None:
                    current_lines.append(clean_line)

        if last_number is not None and current_lines:
            notes_dict[last_number] = '\\n'.join(current_lines)

    return notes_dict


def main(verses_file, notes_file, output_file):
    # Step 1: Extract and clean verses
    df = extract_verses_to_df(verses_file)

    # Step 2: Extract merged notes
    notes_dict = extract_notes(notes_file)

    # Step 3: Add notes column
    df['notes'] = df['number'].map(notes_dict).fillna('')

    # Final columns
    df = df[['number', 'verses', 'notes']]

    # Save as TSV
    df.to_csv(output_file, sep='\t', index=False, encoding='utf-8')

    print(f"✅ Success! TSV file created: '{output_file}'")
    print(f"   Total entries: {len(df)}")
    print("\nPreview of first 10 rows:")
    print(df.head(10).to_string(index=False))


if __name__ == "__main__":
    # Change these filenames as needed
    text1_file = r"Y:\Documents\Python\truyen_kieu\renvois2.txt"                  # First text (will go to Text1 column)
    text2_file = r"Y:\Documents\Python\truyen_kieu\truyen_kieu_complet2.txt"      # Second text (will go to Verses column)
    output_file = r"Y:\Documents\Python\truyen_kieu\merged_notes.tsv"
    main(text2_file, text1_file, output_file)
    

✅ Success! TSV file created: 'Y:\Documents\Python\truyen_kieu\merged_notes.tsv'
   Total entries: 3256

Preview of first 10 rows:
 number                                     verses                                                                                                                                                                                 notes
      1               Trăm năm trong cõi người ta,                                                                                                                                                                                      
      2        Chữ tài chữ mệnh khéo là ghét nhau.                                                     Người có tài thì thường gặp mệnh bạc, hình như Tài, mệnh ghét nhau, xung khắc với nhau, hễ được hơn cái này thì phải kém cái kia.
      3                  Trải qua một cuộc bể dâu,                         Bể dâu: Trong văn chương cổ của chúng ta thường dùng thành ngữ "bãi bể nương dâu", hoặc nói tắt 